# Battaglia12: conditional Fisher forecast versus the saved 9-parameter SBI MAF

This notebook has one purpose: compare a conditional Fisher posterior and the saved nine-parameter NPE at **the same independent Battaglia12 baseline/deproj0 observation**.

The contracts are deliberately strict:

1. The observation is Battaglia12 noise seed 20001 from the 64-profile campaign.
2. The conditional covariance uses the other 63 realizations, noise seeds 20002--20064, with the same signal and mask. The selected observation is never used in the covariance or its mean.
3. Fisher derivatives and all spectra are raw, linearly binned $D_\ell$. The Fisher likelihood is evaluated in this space.
4. SBI alone uses the saved training transform, $x_o=\operatorname{asinh}(D_\ell/s)$.
5. SBI samples are newly drawn from `density_estimator.pkl` at this $x_o$ and restricted to the original hard box prior. No cached posterior samples and no MCMC fallback are used.
6. A Fisher/SBI comparison is made only when the MAF produces a numerically viable fraction of in-prior direct samples. On failure, the notebook saves diagnostics and a clearly labeled Fisher-only corner.

For an old `sbi` run, direct sampling from the saved flow followed by exact box-prior rejection is the mathematical operation performed by `DirectPosterior.sample(..., reject_outside_prior=True)`. It avoids deserializing the version-sensitive `inference.pkl`, while still using the trained density estimator itself.

In [ ]:
from pathlib import Path
import csv
import json
import pickle
import re
import sys
import warnings

EXPECTED_PYTHON = Path("/home/cbllover/miniconda3/envs/halfdome/bin/python").resolve()
ACTIVE_PYTHON = Path(sys.executable).resolve()
if ACTIVE_PYTHON != EXPECTED_PYTHON:
    raise RuntimeError(
        f"Wrong Jupyter kernel: {ACTIVE_PYTHON}. Select 'HalfDome (clean)', "
        f"which runs {EXPECTED_PYTHON}, then restart the kernel and run all cells."
    )

import numpy as np
import matplotlib.pyplot as plt


def find_repo_root(start=Path.cwd()):
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "SBI_analysis").is_dir() and (candidate / "tSZ_visuals").is_dir():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the HalfDome repository.")


REPO_ROOT = find_repo_root()
DATA_ROOT = REPO_ROOT / "SBI_analysis/data_for_cluster/adrian_so_sbi_cases_ell80_7979_dataset_row_sobolrow"

PREPARED_DATASET = DATA_ROOT / "so_masked_baseline_noise_cross_deproj0_ell80_7979_sbi_run.npz"
ENSEMBLE_ROOT = REPO_ROOT / "SBI_analysis/data_for_cluster/ensemble_battaglia12"

NPE_RUN = REPO_ROOT / "SBI_analysis/convergence_tests/N523788"
FISHER_ANALYSIS = REPO_ROOT / "SBI_analysis/adrian_fisher_baseline_deproj0/analysis"
OUTPUT_DIR = REPO_ROOT / "SBI_analysis/adrian_fisher_baseline_deproj0/battaglia12_conditional_fisher_vs_sbi_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PRODUCT = "masked_baseline_noise_cross_deproj0"
MASK_SEED = 12345
ALL_NOISE_SEEDS = tuple(range(20001, 20065))
OBSERVATION_NOISE_SEED = 20001
COVARIANCE_NOISE_SEEDS = tuple(
    seed for seed in ALL_NOISE_SEEDS if seed != OBSERVATION_NOISE_SEED
)

N_PROFILE_EXAMPLES = 12
CONTEXT_REFERENCE_ROWS = 100_000
FISHER_PROPOSAL_DRAWS = 1_000_000
FISHER_SAMPLE_COUNT = 200_000
SBI_SAMPLE_COUNT = 100_000
SBI_PREFLIGHT_DRAWS = 10_000
SBI_PROPOSAL_BATCH = 20_000
SBI_MAX_PROPOSALS = 10_000_000
SBI_MIN_RAW_ACCEPTANCE = 0.01
RANDOM_SEED = 271828

# Cheap controls used before attempting the final posterior draw.
CONTEXT_COVARIANCE_SHRINKAGE = 0.02
N_THETA_NEIGHBORS = 8
FLOW_CONTROL_CONTEXTS = 4
FLOW_CONTROL_DRAWS = 2_000
FLOW_ENSEMBLE_DRAWS_PER_SEED = 256
CONTINUE_DIAGNOSTICS_IF_SBI_INVALID = True

BATTAGLIA12_BY_NAME = {
    "P0": 18.1,
    "xc": 0.497,
    "beta": 4.35,
    "alpha_m_P0": 0.154,
    "alpha_m_xc": -0.00865,
    "alpha_m_beta": 0.0393,
    "alpha_z_P0": -0.758,
    "alpha_z_xc": 0.731,
    "alpha_z_beta": 0.415,
}

LABEL_BY_NAME = {
    "P0": r"P_0",
    "xc": r"x_{\rm c}",
    "beta": r"\beta",
    "alpha_m_P0": r"\alpha_{m,P_0}",
    "alpha_m_xc": r"\alpha_{m,x_{\rm c}}",
    "alpha_m_beta": r"\alpha_{m,\beta}",
    "alpha_z_P0": r"\alpha_{z,P_0}",
    "alpha_z_xc": r"\alpha_{z,x_{\rm c}}",
    "alpha_z_beta": r"\alpha_{z,\beta}",
}

print("Repository:", REPO_ROOT)
print("Prepared dataset:", PREPARED_DATASET)
print("Ensemble root:", ENSEMBLE_ROOT)
print("Observation seed selected from ensemble:", OBSERVATION_NOISE_SEED)
print("NPE run:", NPE_RUN)
print("Outputs:", OUTPUT_DIR)

## Validated inputs

Each realization must include the validator JSON produced by `prepare_validate_battaglia12_sbi_observation.py`. This checks the product, beam/mask/deprojection filename contract, $C_\ell\rightarrow D_\ell$ conversion, binning, and saved NPE transform. The notebook uses the JSON seed metadata rather than filesystem order.

In [ ]:
PROFILE_FILENAME = "battaglia12_masked_baseline_noise_cross_deproj0_binned_dell.npy"
REPORT_FILENAME = "battaglia12_baseline_deproj0_validation.json"


def scalar(value):
    array = np.asarray(value)
    return array.reshape(()).item() if array.size == 1 else value


def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))


def discover_validated_profiles(root):
    root = Path(root).expanduser().resolve()
    if not root.is_dir():
        raise FileNotFoundError(
            f"Missing profile root: {root}\n"
            "Copy the complete cluster output directory, including every prepared/ validation JSON."
        )

    by_seed = {}
    for report_path in sorted(root.rglob(REPORT_FILENAME)):
        report = read_json(report_path)
        contract = report.get("simulator_contract", {})
        seed = int(contract.get("noise_seed", -1))
        mask_seed = int(contract.get("mask_seed", -1))
        profile_path = report_path.parent / PROFILE_FILENAME

        if report.get("status") != "passed" or not report.get("all_checks_passed", False):
            raise RuntimeError(f"Validation did not pass: {report_path}")
        if contract.get("product", PRODUCT) != PRODUCT:
            raise ValueError(f"Wrong product in {report_path}: {contract.get('product')}")
        if mask_seed != MASK_SEED:
            raise ValueError(f"Wrong mask seed {mask_seed} in {report_path}")
        if not profile_path.is_file():
            raise FileNotFoundError(f"Missing validated D_ell vector beside {report_path}: {profile_path}")
        if seed in by_seed:
            raise ValueError(f"Duplicate noise seed {seed}: {by_seed[seed][0]} and {profile_path}")

        values = np.asarray(np.load(profile_path), dtype=np.float64).reshape(-1)
        if not np.all(np.isfinite(values)):
            raise ValueError(f"Non-finite profile: {profile_path}")
        by_seed[seed] = (profile_path, values, report)
    return by_seed


def require_seed_set(by_seed, expected_seeds, label):
    expected = set(map(int, expected_seeds))
    available = set(by_seed)
    missing = sorted(expected - available)
    if missing:
        raise FileNotFoundError(
            f"{label}: missing validated noise seeds {missing}. "
            f"Found seeds: {sorted(available)}"
        )
    extras = sorted(available - expected)
    if extras:
        print(f"{label}: ignoring extra noise seeds {extras}")
    return np.stack([by_seed[seed][1] for seed in expected_seeds], axis=0)


required_files = [
    PREPARED_DATASET,
    NPE_RUN / "density_estimator.pkl",
    NPE_RUN / "x_transform.npz",
    FISHER_ANALYSIS / "derivatives_richardson.npy",
]
for path in required_files:
    if not path.is_file():
        raise FileNotFoundError(path)

with np.load(PREPARED_DATASET, allow_pickle=True) as prepared:
    product = str(scalar(prepared["product"]))
    x_all = np.asarray(prepared["x"], dtype=np.float32)
    theta_all = np.asarray(prepared["theta"], dtype=np.float32)
    ell_binned = np.asarray(prepared["ell_binned"], dtype=np.float64)
    prior_low = np.asarray(prepared["prior_low"], dtype=np.float64)
    prior_high = np.asarray(prepared["prior_high"], dtype=np.float64)
    param_names = [str(item) for item in prepared["param_names"]]

if product != PRODUCT:
    raise ValueError(f"Prepared product is {product!r}, expected {PRODUCT!r}")
if x_all.shape != (theta_all.shape[0], ell_binned.size):
    raise ValueError(f"Prepared x/theta/ell shapes are inconsistent: {x_all.shape}, {theta_all.shape}, {ell_binned.shape}")
if theta_all.shape[1] != len(param_names):
    raise ValueError("Parameter-name count does not match theta columns.")

truth = np.asarray([BATTAGLIA12_BY_NAME[name] for name in param_names], dtype=np.float64)

with np.load(NPE_RUN / "x_transform.npz", allow_pickle=True) as transform_file:
    transform = {key: np.asarray(transform_file[key]).copy() for key in transform_file.files}
transform_mode = str(scalar(transform["mode"])).lower().replace("-", "_")
if transform_mode != "asinh":
    raise ValueError(f"This comparison requires the saved asinh-only run, found {transform_mode!r}")
transform_scale = np.asarray(transform["scale"], dtype=np.float32).reshape(-1)
train_indices = np.asarray(transform["train_indices"], dtype=np.int64).reshape(-1)
if transform_scale.shape != (ell_binned.size,):
    raise ValueError("Saved asinh scale does not match the 40-bin data vector.")
if train_indices.size != 523788:
    warnings.warn(f"Expected 523788 training indices, found {train_indices.size}.")

ensemble_profiles = discover_validated_profiles(ENSEMBLE_ROOT)
all_realizations_dell = require_seed_set(
    ensemble_profiles,
    ALL_NOISE_SEEDS,
    "64-profile Battaglia12 campaign",
)
observation_dell = ensemble_profiles[OBSERVATION_NOISE_SEED][1]
ensemble_dell = np.stack(
    [ensemble_profiles[seed][1] for seed in COVARIANCE_NOISE_SEEDS],
    axis=0,
)
observation_profile_path, _, observation_report = ensemble_profiles[OBSERVATION_NOISE_SEED]
observation_saved_transform_path = observation_profile_path.parent / (
    "battaglia12_masked_baseline_noise_cross_deproj0_transformed.npy"
)
if not observation_saved_transform_path.is_file():
    raise FileNotFoundError(observation_saved_transform_path)

if OBSERVATION_NOISE_SEED in COVARIANCE_NOISE_SEEDS:
    raise RuntimeError("The observation seed must not be included in the covariance ensemble.")
if ensemble_dell.shape != (63, ell_binned.size):
    raise ValueError(f"Expected covariance ensemble shape (63, {ell_binned.size}), got {ensemble_dell.shape}")
if observation_dell.shape != (ell_binned.size,):
    raise ValueError(f"Observation shape mismatch: {observation_dell.shape}")

print("Prepared x:", x_all.shape)
print("Validated covariance ensemble:", ensemble_dell.shape)
print("Leave-one-out observation seed:", OBSERVATION_NOISE_SEED)
print("Parameters:", param_names)

## Spectrum and context checks

The first plot is a direct data-space check. Negative cross-spectrum bins remain negative; no logarithm, clipping, or floor is applied. The second diagnostic checks the joint transformed NPE context. A profile can look normal in every individual bin while still being far outside the learned 40-dimensional correlation structure.

In [ ]:
rng = np.random.default_rng(RANDOM_SEED)
example_indices = rng.choice(train_indices, size=N_PROFILE_EXAMPLES, replace=False)
ensemble_mean = ensemble_dell.mean(axis=0)

fig, ax = plt.subplots(figsize=(7.0, 4.4))
for i, index in enumerate(example_indices):
    ax.plot(
        ell_binned,
        x_all[index],
        color="#4C78A8",
        alpha=0.22,
        lw=0.8,
        label="training profiles" if i == 0 else None,
    )
for i, profile in enumerate(ensemble_dell):
    ax.plot(
        ell_binned,
        profile,
        color="0.55",
        alpha=0.10,
        lw=0.65,
        label="63 covariance realizations" if i == 0 else None,
    )
ax.plot(ell_binned, ensemble_mean, color="black", lw=1.5, label="ensemble mean")
ax.plot(ell_binned, observation_dell, color="#D62728", lw=1.5, label="independent Battaglia12 observation")

positive_scale = np.nanpercentile(np.abs(np.concatenate((ensemble_dell.ravel(), observation_dell))), 20)
ax.set_yscale("symlog", linthresh=max(float(positive_scale) * 0.1, 1e-30))
ax.set_xlabel(r"Multipole, $\ell$")
ax.set_ylabel(r"Binned $D_\ell$")
ax.grid(alpha=0.22)
ax.legend(fontsize=8, ncol=2)
fig.tight_layout()
profile_plot = OUTPUT_DIR / "battaglia12_observation_ensemble_and_training_dell.jpg"
fig.savefig(profile_plot, dpi=300)
plt.show()

x_obs = np.arcsinh(observation_dell.astype(np.float32) / transform_scale)
saved_x_obs = np.asarray(np.load(observation_saved_transform_path), dtype=np.float32).reshape(-1)
if not np.array_equal(x_obs, saved_x_obs) and not np.allclose(x_obs, saved_x_obs, rtol=2e-6, atol=0.0):
    raise ValueError(
        "The independently recomputed asinh observation does not match the validator output."
    )
context_indices = rng.choice(
    train_indices,
    size=min(CONTEXT_REFERENCE_ROWS, train_indices.size),
    replace=False,
)
x_train_context = np.arcsinh(x_all[context_indices] / transform_scale[None, :])
context_mean = x_train_context.mean(axis=0, dtype=np.float64)
context_cov = np.cov(x_train_context.astype(np.float64), rowvar=False)
context_precision = np.linalg.pinv(context_cov, rcond=1e-10, hermitian=True)
context_delta = x_obs.astype(np.float64) - context_mean
obs_mahalanobis2 = float(context_delta @ context_precision @ context_delta)
train_delta = x_train_context.astype(np.float64) - context_mean[None, :]
train_mahalanobis2 = np.einsum("ni,ij,nj->n", train_delta, context_precision, train_delta, optimize=True)
context_percentile = float(np.mean(train_mahalanobis2 <= obs_mahalanobis2))
context_std = x_train_context.std(axis=0, ddof=1, dtype=np.float64)
max_marginal_z = float(np.max(np.abs(context_delta / np.maximum(context_std, 1e-30))))

del x_train_context, train_delta, train_mahalanobis2

print(f"Maximum marginal transformed |z|: {max_marginal_z:.3f}")
print(f"Joint transformed Mahalanobis percentile: {context_percentile:.6f}")
print("A percentile near 1 means the observation is jointly atypical for the NPE training contexts.")

## Expanded profile and joint-support diagnostics

A one-bin envelope is necessary but not sufficient: an observation may be ordinary in every bin while violating correlations among all 40 bins. These checks compare random training profiles, the training 5--95% envelope, the eight training simulations nearest to Battaglia12 in prior-normalized parameter space, and all 64 Battaglia12 noise realizations.

Beam and (f_{\rm sky}) alternatives below are diagnostic hypotheses only. They are never substituted for the validated observation.

In [ ]:
prior_width = prior_high - prior_low
theta_distance = np.sqrt(
    np.mean(
        ((theta_all[train_indices].astype(np.float64) - truth[None, :])
         / prior_width[None, :]) ** 2,
        axis=1,
    )
)
nearest_order = np.argsort(theta_distance)[:N_THETA_NEIGHBORS]
nearest_indices = train_indices[nearest_order]
nearest_distances = theta_distance[nearest_order]

reference_raw = np.asarray(x_all[context_indices], dtype=np.float32)
reference_transformed = np.arcsinh(
    reference_raw / transform_scale[None, :]
).astype(np.float64)
all_battaglia_transformed = np.arcsinh(
    all_realizations_dell.astype(np.float32)
    / transform_scale[None, :]
).astype(np.float64)
nearest_transformed = np.arcsinh(
    x_all[nearest_indices] / transform_scale[None, :]
).astype(np.float64)

reference_mean = reference_transformed.mean(axis=0)
reference_std = np.maximum(
    reference_transformed.std(axis=0, ddof=1), 1e-12
)
reference_standardized = (
    reference_transformed - reference_mean[None, :]
) / reference_std[None, :]
battaglia_standardized = (
    all_battaglia_transformed - reference_mean[None, :]
) / reference_std[None, :]

standardized_covariance = np.cov(
    reference_standardized, rowvar=False
)
standardized_covariance = (
    (1.0 - CONTEXT_COVARIANCE_SHRINKAGE)
    * standardized_covariance
    + CONTEXT_COVARIANCE_SHRINKAGE
    * np.diag(np.diag(standardized_covariance))
)
context_eigenvalues, context_eigenvectors = np.linalg.eigh(
    standardized_covariance
)
context_floor = max(
    float(context_eigenvalues.max()) * 1e-8, 1e-10
)
context_inverse = (
    context_eigenvectors
    * (1.0 / np.maximum(context_eigenvalues, context_floor))
) @ context_eigenvectors.T

reference_mahalanobis2 = np.einsum(
    "ni,ij,nj->n",
    reference_standardized,
    context_inverse,
    reference_standardized,
    optimize=True,
)
battaglia_mahalanobis2 = np.einsum(
    "ni,ij,nj->n",
    battaglia_standardized,
    context_inverse,
    battaglia_standardized,
    optimize=True,
)
battaglia_mahalanobis_percentiles = np.asarray(
    [
        np.mean(reference_mahalanobis2 <= value)
        for value in battaglia_mahalanobis2
    ],
    dtype=np.float64,
)
observation_marginal_z = battaglia_standardized[0]
ensemble_mean_context = all_battaglia_transformed.mean(axis=0)
ensemble_mean_marginal_z = (
    ensemble_mean_context - reference_mean
) / reference_std

beam_sigma_rad = (
    np.deg2rad(2.0 / 60.0) / np.sqrt(8.0 * np.log(2.0))
)
beam_power = np.exp(
    -ell_binned * (ell_binned + 1.0) * beam_sigma_rad**2
)
contract_hypotheses = {
    "original": observation_dell,
    "divide_fsky": observation_dell / 0.4,
    "multiply_fsky": observation_dell * 0.4,
    "deconvolve_2arcmin_beam_power":
        observation_dell / beam_power,
    "apply_2arcmin_beam_power_again":
        observation_dell * beam_power,
}
hypothesis_rows = []
for name, raw_profile in contract_hypotheses.items():
    transformed_profile = np.arcsinh(
        raw_profile.astype(np.float32) / transform_scale
    ).astype(np.float64)
    standardized_profile = (
        transformed_profile - reference_mean
    ) / reference_std
    distance2 = float(
        standardized_profile @ context_inverse
        @ standardized_profile
    )
    hypothesis_rows.append(
        {
            "name": name,
            "mahalanobis2": distance2,
            "mahalanobis_percentile": float(
                np.mean(reference_mahalanobis2 <= distance2)
            ),
            "max_abs_marginal_z": float(
                np.max(np.abs(standardized_profile))
            ),
        }
    )

raw_quantiles = np.quantile(
    reference_raw, [0.05, 0.5, 0.95], axis=0
)
transformed_quantiles = np.quantile(
    reference_transformed, [0.05, 0.5, 0.95], axis=0
)
fig, axes = plt.subplots(2, 2, figsize=(11.2, 8.0))

ax = axes[0, 0]
ax.fill_between(
    ell_binned, raw_quantiles[0], raw_quantiles[2],
    color="#4C78A8", alpha=0.16, label="training 5--95%",
)
ax.plot(
    ell_binned, raw_quantiles[1], color="#4C78A8",
    lw=1.2, label="training median",
)
for i, index in enumerate(example_indices):
    ax.plot(
        ell_binned, x_all[index], color="#4C78A8",
        alpha=0.16, lw=0.65,
        label="random training profiles" if i == 0 else None,
    )
for i, index in enumerate(nearest_indices):
    ax.plot(
        ell_binned, x_all[index], color="#9467BD",
        alpha=0.25, lw=0.75,
        label=r"nearest training $\theta$" if i == 0 else None,
    )
for i, profile in enumerate(all_realizations_dell):
    ax.plot(
        ell_binned, profile, color="0.55",
        alpha=0.07, lw=0.55,
        label="64 Battaglia12 seeds" if i == 0 else None,
    )
ax.plot(
    ell_binned, all_realizations_dell.mean(axis=0),
    color="black", lw=1.4, label="Battaglia12 ensemble mean",
)
ax.plot(
    ell_binned, observation_dell, color="#D62728", lw=1.35,
    label=f"Battaglia12 seed {OBSERVATION_NOISE_SEED}",
)
raw_linthresh = max(
    float(np.nanpercentile(np.abs(reference_raw), 20)) * 0.1,
    1e-30,
)
ax.set_yscale("symlog", linthresh=raw_linthresh)
ax.set_xlabel(r"Multipole, $\ell$")
ax.set_ylabel(r"Binned $D_\ell$")
ax.grid(alpha=0.2)
ax.legend(frameon=False, fontsize=6.2, ncol=2)

ax = axes[0, 1]
ax.fill_between(
    ell_binned, transformed_quantiles[0],
    transformed_quantiles[2], color="#4C78A8",
    alpha=0.16, label="training 5--95%",
)
ax.plot(
    ell_binned, transformed_quantiles[1], color="#4C78A8",
    lw=1.2, label="training median",
)
for i, profile in enumerate(nearest_transformed):
    ax.plot(
        ell_binned, profile, color="#9467BD",
        alpha=0.25, lw=0.75,
        label=r"nearest training $\theta$" if i == 0 else None,
    )
for profile in all_battaglia_transformed:
    ax.plot(
        ell_binned, profile, color="0.55",
        alpha=0.07, lw=0.55,
    )
ax.plot(
    ell_binned, ensemble_mean_context, color="black",
    lw=1.4, label="Battaglia12 ensemble mean",
)
ax.plot(
    ell_binned, x_obs, color="#D62728", lw=1.35,
    label=f"Battaglia12 seed {OBSERVATION_NOISE_SEED}",
)
ax.set_xlabel(r"Multipole, $\ell$")
ax.set_ylabel(r"$\operatorname{asinh}(D_\ell/s_\ell)$")
ax.grid(alpha=0.2)
ax.legend(frameon=False, fontsize=6.2, ncol=2)

ax = axes[1, 0]
ax.axhspan(
    -3.0, 3.0, color="0.85", alpha=0.4, label=r"$|z|<3$"
)
ax.axhline(0.0, color="black", lw=0.7)
ax.plot(
    ell_binned, observation_marginal_z, color="#D62728",
    marker="o", ms=2.5, lw=0.9,
    label=f"seed {OBSERVATION_NOISE_SEED}",
)
ax.plot(
    ell_binned, ensemble_mean_marginal_z, color="black",
    lw=1.1, label="ensemble mean",
)
ax.set_xlabel(r"Multipole, $\ell$")
ax.set_ylabel("Marginal transformed $z$")
ax.grid(alpha=0.2)
ax.legend(frameon=False, fontsize=7)

ax = axes[1, 1]
positive_training = reference_mahalanobis2[
    reference_mahalanobis2 > 0.0
]
distance_min = max(
    float(positive_training.min()),
    min(float(battaglia_mahalanobis2.min()), 1.0),
)
distance_max = max(
    float(reference_mahalanobis2.max()),
    float(battaglia_mahalanobis2.max()),
)
bins = np.geomspace(distance_min, distance_max * 1.01, 70)
ax.hist(
    reference_mahalanobis2, bins=bins, density=True,
    color="#4C78A8", alpha=0.55, label="training contexts",
)
for i, value in enumerate(battaglia_mahalanobis2):
    ax.axvline(
        value, color="#D62728", alpha=0.10, lw=0.65,
        label="64 Battaglia12 seeds" if i == 0 else None,
    )
ax.axvline(
    battaglia_mahalanobis2[0], color="#8B0000", lw=1.3,
    label=f"seed {OBSERVATION_NOISE_SEED}",
)
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("Shrinkage Mahalanobis distance squared")
ax.set_ylabel("Density")
ax.grid(alpha=0.2)
ax.legend(frameon=False, fontsize=7)

fig.tight_layout()
profile_context_plot = (
    OUTPUT_DIR / "battaglia12_training_profile_and_context_sanity.jpg"
)
fig.savefig(profile_context_plot, dpi=300)
plt.show()

profile_context_report = {
    "training_reference_rows": int(reference_transformed.shape[0]),
    "nearest_theta_dataset_indices": nearest_indices.tolist(),
    "nearest_theta_prior_normalized_rms_distances":
        nearest_distances.tolist(),
    "observation_max_abs_marginal_z": float(
        np.max(np.abs(observation_marginal_z))
    ),
    "ensemble_mean_max_abs_marginal_z": float(
        np.max(np.abs(ensemble_mean_marginal_z))
    ),
    "battaglia_mahalanobis2_min_median_max": [
        float(np.min(battaglia_mahalanobis2)),
        float(np.median(battaglia_mahalanobis2)),
        float(np.max(battaglia_mahalanobis2)),
    ],
    "battaglia_mahalanobis_percentile_min_median_max": [
        float(np.min(battaglia_mahalanobis_percentiles)),
        float(np.median(battaglia_mahalanobis_percentiles)),
        float(np.max(battaglia_mahalanobis_percentiles)),
    ],
    "context_covariance_shrinkage":
        CONTEXT_COVARIANCE_SHRINKAGE,
    "contract_hypotheses": hypothesis_rows,
    "figure": str(profile_context_plot),
}
(
    OUTPUT_DIR / "battaglia12_profile_context_sanity.json"
).write_text(
    json.dumps(profile_context_report, indent=2) + "\n",
    encoding="utf-8",
)

print("Nearest theta distances:", nearest_distances)
print(
    "Battaglia12 joint percentiles (min/median/max):",
    profile_context_report[
        "battaglia_mahalanobis_percentile_min_median_max"
    ],
)
for row in hypothesis_rows:
    print(
        f"{row['name']}: percentile="
        f"{row['mahalanobis_percentile']:.6f}, "
        f"max |z|={row['max_abs_marginal_z']:.3f}"
    )
print("Saved:", profile_context_plot)

## Conditional-noise covariance

All 64 simulations have the same Battaglia12 signal and the same apodized mask. Seed 20001 is reserved as the observation; only the other 63 enter the covariance. Only the two split-noise realizations change. Their scatter therefore estimates

$$C_{bb'}^{\rm cond}=\operatorname{Cov}\!\left(\hat D_b,\hat D_{b'}\mid \theta_{\rm B12},\,\mathrm{signal},\,\mathrm{mask}\right).$$

There are only 63 covariance realizations for 40 bins. Directly inverting the raw sample covariance would have a noisy, biased precision matrix. The code standardizes each bin, applies Oracle Approximating Shrinkage (OAS) to the correlation-scale covariance, and then restores the measured bin variances. This preserves the empirical heteroscedastic noise while regularizing poorly measured cross-bin correlations.

In [ ]:
def oas_covariance_standardized(samples):
    values = np.asarray(samples, dtype=np.float64)
    if values.ndim != 2:
        raise ValueError("OAS input must be (n_realizations, n_features).")
    centered = values - values.mean(axis=0, keepdims=True)
    n_samples, n_features = centered.shape
    empirical = centered.T @ centered / float(n_samples)
    mu = float(np.trace(empirical) / n_features)
    alpha = float(np.mean(empirical**2))
    denominator = (n_samples + 1.0) * (alpha - mu**2 / n_features)
    shrinkage = 1.0 if denominator <= 0.0 else min((alpha + mu**2) / denominator, 1.0)
    shrunk = (1.0 - shrinkage) * empirical
    shrunk.flat[:: n_features + 1] += shrinkage * mu
    return 0.5 * (shrunk + shrunk.T), float(shrinkage)


def stable_inverse(matrix, relative_floor=1e-12):
    symmetric = 0.5 * (np.asarray(matrix, dtype=np.float64) + np.asarray(matrix, dtype=np.float64).T)
    eigenvalues, eigenvectors = np.linalg.eigh(symmetric)
    floor = max(float(eigenvalues.max()) * relative_floor, 1e-300)
    if eigenvalues.min() <= 0.0:
        raise ValueError(f"Covariance is not positive definite: lambda_min={eigenvalues.min():.4e}")
    inverse = (eigenvectors * (1.0 / np.maximum(eigenvalues, floor))) @ eigenvectors.T
    return inverse, eigenvalues


conditional_residuals = ensemble_dell - ensemble_mean[None, :]
bin_scale = conditional_residuals.std(axis=0, ddof=1)
if np.any(~np.isfinite(bin_scale)) or np.any(bin_scale <= 0.0):
    raise ValueError("Every D_ell bin must have a positive finite ensemble standard deviation.")
standardized_residuals = conditional_residuals / bin_scale[None, :]
shrunk_standard_cov, oas_shrinkage = oas_covariance_standardized(standardized_residuals)
conditional_covariance = bin_scale[:, None] * shrunk_standard_cov * bin_scale[None, :]
conditional_precision, covariance_eigenvalues = stable_inverse(conditional_covariance)
conditional_correlation = conditional_covariance / np.sqrt(
    np.diag(conditional_covariance)[:, None] * np.diag(conditional_covariance)[None, :]
)

np.save(OUTPUT_DIR / "conditional_noise_covariance_leave_one_out_63_oas.npy", conditional_covariance)
np.save(OUTPUT_DIR / "conditional_noise_precision_leave_one_out_63_oas.npy", conditional_precision)

fig, axes = plt.subplots(1, 2, figsize=(8.0, 3.35))
image = axes[0].imshow(conditional_correlation, origin="lower", vmin=-1, vmax=1, cmap="RdBu_r")
axes[0].set_title("Conditional-noise correlation")
axes[0].set_xlabel("$D_\\ell$ bin")
axes[0].set_ylabel("$D_\\ell$ bin")
fig.colorbar(image, ax=axes[0], fraction=0.046, pad=0.04)
axes[1].semilogy(np.arange(1, len(covariance_eigenvalues) + 1), covariance_eigenvalues[::-1], "o-", ms=3)
axes[1].set_title("Regularized covariance spectrum")
axes[1].set_xlabel("Ordered mode")
axes[1].set_ylabel("Eigenvalue")
axes[1].grid(alpha=0.25)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "conditional_noise_covariance_diagnostics.jpg", dpi=300)
plt.show()

print(f"OAS shrinkage in standardized bin space: {oas_shrinkage:.4f}")
print(f"Covariance condition number: {covariance_eigenvalues.max() / covariance_eigenvalues.min():.4e}")

## Fisher posterior at the independent observation

Let $J_{ib}=\partial D_b/\partial\theta_i$ be the saved Richardson derivative matrix. The local linear model is

$$m(\theta)=\bar D_{\rm ens}+J^T(\theta-\theta_{\rm B12}),$$

and the likelihood uses the independent observation and the regularized conditional covariance. The ensemble mean is used as the finite-ensemble estimate of the split-cross-spectrum expectation at Battaglia12. The posterior is this Gaussian linear likelihood multiplied by the same hard uniform prior used for SBI. Importance sampling handles weak or rank-deficient Fisher directions without pretending that an unconstrained inverse Fisher matrix is a physical posterior.

In [ ]:
def systematic_resample(samples, weights, count, rng):
    positions = (rng.random() + np.arange(count)) / count
    cumulative = np.cumsum(weights)
    cumulative[-1] = 1.0
    indices = np.searchsorted(cumulative, positions, side="right")
    return np.asarray(samples[indices], dtype=np.float64)


def sample_fisher_hard_prior(
    derivatives,
    precision,
    observation,
    model_at_truth,
    truth,
    prior_low,
    prior_high,
    proposal_draws,
    sample_count,
    seed,
):
    derivatives = np.asarray(derivatives, dtype=np.float64)
    residual = np.asarray(observation, dtype=np.float64) - np.asarray(model_at_truth, dtype=np.float64)
    fisher = derivatives @ precision @ derivatives.T
    fisher = 0.5 * (fisher + fisher.T)
    eigenvalues = np.linalg.eigvalsh(fisher)
    threshold = max(float(eigenvalues.max()) * 1e-8, 0.0)
    rank = int(np.count_nonzero(eigenvalues > threshold))

    prior_width = prior_high - prior_low
    gaussian_prior_sigma = prior_width / np.sqrt(12.0)
    gaussian_prior_precision = np.diag(1.0 / gaussian_prior_sigma**2)
    proposal_precision = fisher + gaussian_prior_precision
    proposal_covariance, _ = stable_inverse(proposal_precision)
    score = derivatives @ precision @ residual
    proposal_mean = truth + proposal_covariance @ score

    rng = np.random.default_rng(seed)
    accepted = []
    drawn = 0
    while drawn < int(proposal_draws):
        count = min(100_000, int(proposal_draws) - drawn)
        batch = rng.multivariate_normal(proposal_mean, proposal_covariance, size=count)
        inside = np.all((batch >= prior_low[None, :]) & (batch <= prior_high[None, :]), axis=1)
        if np.any(inside):
            accepted.append(batch[inside])
        drawn += count
    if not accepted:
        raise RuntimeError("Fisher importance proposal produced no in-prior points.")

    candidates = np.concatenate(accepted, axis=0)
    standardized = (candidates - truth[None, :]) / gaussian_prior_sigma[None, :]
    log_weight = 0.5 * np.sum(standardized**2, axis=1)
    log_weight -= log_weight.max()
    weights = np.exp(log_weight)
    weights /= weights.sum()
    ess = float(1.0 / np.sum(weights**2))
    if ess < 5_000:
        raise RuntimeError(f"Fisher importance ESS={ess:.1f}; increase FISHER_PROPOSAL_DRAWS.")

    samples = systematic_resample(candidates, weights, int(sample_count), rng)
    return samples, {
        "fisher_matrix": fisher,
        "fisher_eigenvalues": eigenvalues,
        "fisher_rank": rank,
        "rank_threshold": threshold,
        "residual": residual,
        "proposal_mean": proposal_mean,
        "proposal_covariance": proposal_covariance,
        "proposal_inside_prior_fraction": float(candidates.shape[0] / proposal_draws),
        "importance_ess": ess,
    }


derivatives = np.asarray(np.load(FISHER_ANALYSIS / "derivatives_richardson.npy"), dtype=np.float64)
if derivatives.shape != (len(param_names), ell_binned.size):
    raise ValueError(f"Derivative shape {derivatives.shape} is incompatible with {(len(param_names), ell_binned.size)}")

fisher_samples, fisher_info = sample_fisher_hard_prior(
    derivatives=derivatives,
    precision=conditional_precision,
    observation=observation_dell,
    model_at_truth=ensemble_mean,
    truth=truth,
    prior_low=prior_low,
    prior_high=prior_high,
    proposal_draws=FISHER_PROPOSAL_DRAWS,
    sample_count=FISHER_SAMPLE_COUNT,
    seed=RANDOM_SEED,
)
np.save(OUTPUT_DIR / "fisher_conditional_battaglia12_samples.npy", fisher_samples)
np.save(OUTPUT_DIR / "fisher_conditional_matrix.npy", fisher_info["fisher_matrix"])

print(f"Fisher numerical rank: {fisher_info['fisher_rank']}/{len(param_names)}")
print(f"Fisher proposal in-prior fraction: {fisher_info['proposal_inside_prior_fraction']:.3%}")
print(f"Fisher importance ESS: {fisher_info['importance_ess']:.1f}")

## Fresh SBI samples at the same $x_o$

The effective operation is

```python
x_o = np.arcsinh(observation_dell / saved_scale)
raw = density_estimator.sample(num_proposals, context=x_o)
samples = raw[inside_original_uniform_prior]
```

This is the direct-NPE equivalent of

```python
posterior = inference.build_posterior(density_estimator)
samples = posterior.sample((num_samples,), x=x_o)
```

for a `DirectPosterior` with its default hard-prior rejection. Sampling the saved estimator directly avoids old/new `sbi` pickle API conflicts; it does **not** alter the learned density and it does not replace failed rejection with MCMC.

In [ ]:
def normalize_flow_samples(samples, n_params):
    import torch

    if torch.is_tensor(samples):
        array = samples.detach().cpu().numpy()
    else:
        array = np.asarray(samples)
    while array.ndim > 2 and array.shape[0] == 1:
        array = array[0]
    if array.ndim == 1 and array.size == n_params:
        array = array.reshape(1, n_params)
    if array.ndim != 2:
        raise ValueError(
            f"Unexpected density-estimator sample shape: {array.shape}"
        )
    if array.shape[1] != n_params and array.shape[0] == n_params:
        array = array.T
    if array.shape[1] != n_params:
        raise ValueError(
            f"Flow returned shape {array.shape}; "
            f"expected {n_params} parameters."
        )
    return np.asarray(array, dtype=np.float64)


def draw_flow(density_estimator, context, count):
    import torch

    context = torch.as_tensor(
        context, dtype=torch.float32
    ).reshape(1, -1)
    with torch.no_grad():
        try:
            values = density_estimator.sample(
                int(count), context=context
            )
        except TypeError:
            values = density_estimator.sample(
                (int(count),), context=context
            )
    return normalize_flow_samples(values, len(param_names))


def in_prior(samples):
    return np.all(
        (samples >= prior_low[None, :])
        & (samples <= prior_high[None, :]),
        axis=1,
    )


def raw_flow_summary(
    density_estimator, label, context, count
):
    samples = draw_flow(density_estimator, context, count)
    inside_by_parameter = (
        (samples >= prior_low[None, :])
        & (samples <= prior_high[None, :])
    )
    return samples, {
        "label": label,
        "draws": int(samples.shape[0]),
        "all_parameter_prior_acceptance": float(
            np.mean(np.all(inside_by_parameter, axis=1))
        ),
        "finite_fraction": float(
            np.mean(np.all(np.isfinite(samples), axis=1))
        ),
    }


def sample_saved_direct_posterior(
    density_estimator, context, target_count
):
    accepted_batches = []
    accepted_count = 0
    proposal_count = 0
    while (
        accepted_count < int(target_count)
        and proposal_count < SBI_MAX_PROPOSALS
    ):
        request = min(
            SBI_PROPOSAL_BATCH,
            SBI_MAX_PROPOSALS - proposal_count,
        )
        raw = draw_flow(density_estimator, context, request)
        accepted = raw[in_prior(raw)]
        if accepted.size:
            accepted_batches.append(accepted)
            accepted_count += accepted.shape[0]
        proposal_count += raw.shape[0]
        if proposal_count % (10 * SBI_PROPOSAL_BATCH) == 0:
            print(
                f"proposals={proposal_count:,}, "
                f"accepted={accepted_count:,}"
            )
    if accepted_count < int(target_count):
        raise RuntimeError(
            f"Only {accepted_count:,}/{target_count:,} in-prior "
            f"samples after {proposal_count:,} direct MAF proposals."
        )
    return (
        np.concatenate(accepted_batches, axis=0)[:int(target_count)],
        proposal_count,
    )


import torch

try:
    torch.set_num_threads(
        max(1, min(8, torch.get_num_threads()))
    )
except RuntimeError:
    pass

with open(NPE_RUN / "density_estimator.pkl", "rb") as handle:
    density_estimator = pickle.load(handle)
density_estimator.eval()

# Positive controls distinguish a broken flow/API from an invalid context.
saved_run_context = np.asarray(
    np.load(NPE_RUN / "obs_transformed.npy"),
    dtype=np.float32,
).reshape(-1)
flow_control_rows = []
_, saved_control = raw_flow_summary(
    density_estimator,
    "saved dataset-row observation",
    saved_run_context,
    FLOW_CONTROL_DRAWS,
)
flow_control_rows.append(saved_control)

control_indices = rng.choice(
    train_indices,
    size=min(FLOW_CONTROL_CONTEXTS, train_indices.size),
    replace=False,
)
for index in control_indices:
    transformed_context = np.arcsinh(
        x_all[index].astype(np.float32) / transform_scale
    )
    _, row = raw_flow_summary(
        density_estimator,
        f"training row {int(index)}",
        transformed_context,
        FLOW_CONTROL_DRAWS,
    )
    flow_control_rows.append(row)

# Test all noise seeds to identify single-seed versus systematic failure.
battaglia_flow_acceptance = []
for seed, transformed_context in zip(
    ALL_NOISE_SEEDS, all_battaglia_transformed
):
    _, row = raw_flow_summary(
        density_estimator,
        f"Battaglia12 seed {seed}",
        transformed_context,
        FLOW_ENSEMBLE_DRAWS_PER_SEED,
    )
    battaglia_flow_acceptance.append(
        row["all_parameter_prior_acceptance"]
    )

raw_preflight, battaglia_preflight = raw_flow_summary(
    density_estimator,
    f"Battaglia12 seed {OBSERVATION_NOISE_SEED} preflight",
    x_obs,
    SBI_PREFLIGHT_DRAWS,
)
raw_acceptance = battaglia_preflight[
    "all_parameter_prior_acceptance"
]
expected_proposals = (
    SBI_SAMPLE_COUNT / max(raw_acceptance, 1e-300)
)

inside_by_parameter = (
    (raw_preflight >= prior_low[None, :])
    & (raw_preflight <= prior_high[None, :])
)
flow_quantiles = np.quantile(
    raw_preflight, [0.01, 0.5, 0.99], axis=0
)
parameter_flow_rows = []
for index, name in enumerate(param_names):
    parameter_flow_rows.append(
        {
            "parameter": name,
            "prior_low": float(prior_low[index]),
            "prior_high": float(prior_high[index]),
            "inside_prior_fraction": float(
                inside_by_parameter[:, index].mean()
            ),
            "raw_q01": float(flow_quantiles[0, index]),
            "raw_median": float(flow_quantiles[1, index]),
            "raw_q99": float(flow_quantiles[2, index]),
        }
    )

control_csv = (
    OUTPUT_DIR / "sbi_flow_context_acceptance_controls.csv"
)
with control_csv.open(
    "w", newline="", encoding="utf-8"
) as handle:
    writer = csv.DictWriter(
        handle, fieldnames=list(flow_control_rows[0])
    )
    writer.writeheader()
    writer.writerows(flow_control_rows)

parameter_csv = (
    OUTPUT_DIR / "sbi_battaglia_raw_parameter_diagnostics.csv"
)
with parameter_csv.open(
    "w", newline="", encoding="utf-8"
) as handle:
    writer = csv.DictWriter(
        handle, fieldnames=list(parameter_flow_rows[0])
    )
    writer.writeheader()
    writer.writerows(parameter_flow_rows)

fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.2))
control_acceptance = [
    row["all_parameter_prior_acceptance"]
    for row in flow_control_rows
]
axes[0].scatter(
    np.zeros(len(control_acceptance)),
    control_acceptance,
    color="#4C78A8",
    s=28,
    alpha=0.75,
    label="saved/training contexts",
)
axes[0].scatter(
    np.ones(len(battaglia_flow_acceptance)),
    battaglia_flow_acceptance,
    color="#D62728",
    s=18,
    alpha=0.55,
    label="64 Battaglia12 seeds",
)
axes[0].set_xticks([0, 1])
axes[0].set_xticklabels(
    ["in-distribution controls", "Battaglia12 ensemble"]
)
axes[0].set_ylabel(
    "Raw flow fraction inside all prior bounds"
)
axes[0].set_ylim(-0.02, 1.02)
axes[0].grid(axis="y", alpha=0.2)
axes[0].legend(frameon=False, fontsize=7)

prior_width_plot = prior_high - prior_low
q01_position = (
    flow_quantiles[0] - prior_low
) / prior_width_plot
median_position = (
    flow_quantiles[1] - prior_low
) / prior_width_plot
q99_position = (
    flow_quantiles[2] - prior_low
) / prior_width_plot
y = np.arange(len(param_names))
axes[1].hlines(
    y, q01_position, q99_position,
    color="#C44E52", lw=2.0, alpha=0.75,
)
axes[1].scatter(
    median_position, y, color="#8B0000", s=18, zorder=3
)
axes[1].axvspan(
    0.0, 1.0, color="0.85", alpha=0.45
)
axes[1].axvline(
    0.0, color="black", lw=0.7, ls="--"
)
axes[1].axvline(
    1.0, color="black", lw=0.7, ls="--"
)
axes[1].set_yticks(y)
axes[1].set_yticklabels(
    [
        "$" + LABEL_BY_NAME.get(name, name) + "$"
        for name in param_names
    ]
)
axes[1].set_xlabel(
    "Raw-flow position relative to prior (1--99%, median)"
)
axes[1].grid(axis="x", alpha=0.2)

fig.tight_layout()
flow_diagnostic_plot = (
    OUTPUT_DIR / "sbi_flow_support_sanity_checks.jpg"
)
fig.savefig(flow_diagnostic_plot, dpi=300)
plt.show()

control_acceptance_median = float(
    np.median(control_acceptance)
)
battaglia_acceptance_max = float(
    np.max(battaglia_flow_acceptance)
)
flow_is_healthy_on_controls = (
    control_acceptance_median >= SBI_MIN_RAW_ACCEPTANCE
)
sbi_conditioning_is_viable = (
    raw_acceptance >= SBI_MIN_RAW_ACCEPTANCE
    and expected_proposals <= SBI_MAX_PROPOSALS
)

flow_diagnostic_report = {
    "control_contexts": flow_control_rows,
    "control_acceptance_median": control_acceptance_median,
    "flow_is_healthy_on_controls": flow_is_healthy_on_controls,
    "battaglia_seed_acceptance_min_median_max": [
        float(np.min(battaglia_flow_acceptance)),
        float(np.median(battaglia_flow_acceptance)),
        battaglia_acceptance_max,
    ],
    "battaglia_seeds_with_any_accepted_sample": int(
        np.count_nonzero(
            np.asarray(battaglia_flow_acceptance) > 0.0
        )
    ),
    "selected_observation_raw_acceptance": raw_acceptance,
    "estimated_proposals_for_requested_samples":
        expected_proposals,
    "sbi_conditioning_is_viable":
        sbi_conditioning_is_viable,
    "parameter_diagnostics": parameter_flow_rows,
    "figure": str(flow_diagnostic_plot),
}
flow_report_path = (
    OUTPUT_DIR / "sbi_flow_support_sanity_checks.json"
)
flow_report_path.write_text(
    json.dumps(flow_diagnostic_report, indent=2) + "\n",
    encoding="utf-8",
)

print(
    "Control raw-prior acceptance:",
    [f"{value:.3%}" for value in control_acceptance],
)
print(
    "Battaglia12 64-seed acceptance (min/median/max):",
    [
        f"{np.min(battaglia_flow_acceptance):.3%}",
        f"{np.median(battaglia_flow_acceptance):.3%}",
        f"{np.max(battaglia_flow_acceptance):.3%}",
    ],
)
print(
    f"Selected Battaglia12 raw acceptance: "
    f"{raw_acceptance:.6%}"
)

if not flow_is_healthy_on_controls:
    raise RuntimeError(
        "The flow also fails at known dataset/training contexts. "
        "This indicates an estimator deserialization or sampling-API "
        "problem rather than a Battaglia12 context problem."
    )

sbi_samples = None
sbi_proposals = 0
sbi_failure_reason = None
if sbi_conditioning_is_viable:
    sbi_samples, sbi_proposals = sample_saved_direct_posterior(
        density_estimator, x_obs, SBI_SAMPLE_COUNT
    )
    np.save(
        OUTPUT_DIR / "sbi_maf_N523788_battaglia12_samples.npy",
        sbi_samples,
    )
    print(
        f"Saved {len(sbi_samples):,} fresh SBI samples from "
        f"{sbi_proposals:,} raw proposals."
    )
else:
    sbi_failure_reason = (
        "The flow is healthy on saved/training contexts, but all "
        "tested Battaglia12 contexts drive the conditional density "
        "outside the original hard prior. This is a systematic "
        "context/simulator mismatch, not a rejection-sampler fault."
    )
    print("SBI COMPARISON DISABLED:", sbi_failure_reason)
    if not CONTINUE_DIAGNOSTICS_IF_SBI_INVALID:
        raise RuntimeError(sbi_failure_reason)

## Direct nine-parameter comparison when SBI support is valid

Fisher is shown as blue dashed 68/95% contours. Fresh SBI is the final sample set and is the only filled posterior. Black dotted lines mark the Battaglia12 truth. Both posteriors use the same parameter order, bounds, observation, and binned $D_\ell$ contract.

In [ ]:
try:
    from getdist import MCSamples, plots
except ModuleNotFoundError:
    MCSamples = None
    plots = None



def summarize(samples):
    values = np.asarray(samples, dtype=np.float64)
    return (
        values.mean(axis=0),
        values.std(axis=0, ddof=1),
    )


fisher_mean, fisher_std = summarize(fisher_samples)
if sbi_samples is None:
    sbi_mean = np.full(len(param_names), np.nan)
    sbi_std = np.full(len(param_names), np.nan)
else:
    sbi_mean, sbi_std = summarize(sbi_samples)

summary_csv = (
    OUTPUT_DIR
    / "fisher_conditional_vs_sbi_battaglia12_summary.csv"
)
with summary_csv.open(
    "w", newline="", encoding="utf-8"
) as handle:
    writer = csv.writer(handle)
    writer.writerow(
        [
            "parameter",
            "truth",
            "prior_low",
            "prior_high",
            "fisher_mean",
            "fisher_std",
            "sbi_mean",
            "sbi_std",
        ]
    )
    for row in zip(
        param_names,
        truth,
        prior_low,
        prior_high,
        fisher_mean,
        fisher_std,
        sbi_mean,
        sbi_std,
    ):
        writer.writerow(row)

comparison_status = (
    "fisher_and_sbi_comparison"
    if sbi_samples is not None
    else "diagnostic_only_sbi_conditioning_invalid"
)
summary_json = {
    "status": comparison_status,
    "product": PRODUCT,
    "mask_seed": MASK_SEED,
    "all_noise_seeds": list(ALL_NOISE_SEEDS),
    "covariance_noise_seeds": list(
        COVARIANCE_NOISE_SEEDS
    ),
    "observation_noise_seed": OBSERVATION_NOISE_SEED,
    "observation_excluded_from_covariance_and_mean": True,
    "covariance": (
        "same-signal/same-mask conditional noise; "
        "standardized-bin OAS"
    ),
    "oas_shrinkage": oas_shrinkage,
    "context_mahalanobis_percentile": float(
        battaglia_mahalanobis_percentiles[0]
    ),
    "context_max_abs_marginal_z": float(
        np.max(np.abs(observation_marginal_z))
    ),
    "sbi_flow_healthy_on_controls":
        flow_is_healthy_on_controls,
    "sbi_raw_preflight_acceptance": raw_acceptance,
    "sbi_all_64_seed_acceptance_max":
        battaglia_acceptance_max,
    "sbi_raw_proposals": sbi_proposals,
    "sbi_failure_reason": sbi_failure_reason,
    "fisher_rank": fisher_info["fisher_rank"],
    "fisher_importance_ess":
        fisher_info["importance_ess"],
    "outputs": {
        "profile_context_diagnostics":
            str(profile_context_plot),
        "flow_support_diagnostics":
            str(flow_diagnostic_plot),
        "fisher_samples": str(
            OUTPUT_DIR
            / "fisher_conditional_battaglia12_samples.npy"
        ),
        "sbi_samples": (
            str(
                OUTPUT_DIR
                / "sbi_maf_N523788_battaglia12_samples.npy"
            )
            if sbi_samples is not None
            else None
        ),
        "summary_csv": str(summary_csv),
    },
}
summary_json_path = (
    OUTPUT_DIR
    / "fisher_conditional_vs_sbi_battaglia12_summary.json"
)
summary_json_path.write_text(
    json.dumps(summary_json, indent=2) + "\n",
    encoding="utf-8",
)

if MCSamples is None:
    corner_path = None
    print("Status:", comparison_status)
    print("Saved summary:", summary_csv)
    print("Saved audit:", summary_json_path)
    print(
        "GetDist is unavailable in this Jupyter kernel, so the "
        "corner was skipped. Install it into this exact kernel with:\n"
        f"  {sys.executable} -m pip install getdist"
    )
else:
    names = [f"p{i}" for i in range(len(param_names))]
    labels = [
        LABEL_BY_NAME.get(name, name) for name in param_names
    ]
    ranges = {
        name: (prior_low[i], prior_high[i])
        for i, name in enumerate(names)
    }

    fisher_gd = MCSamples(
        samples=fisher_samples,
        names=names,
        labels=labels,
        label="Fisher: conditional noise (68/95%)",
        ranges=ranges,
    )
    plot_samples = [fisher_gd]
    filled = [False]
    legend_labels = ["Fisher: conditional noise"]
    contour_colors = ["#2F5D7C"]
    line_args = [
        {"color": "#2F5D7C", "lw": 1.2, "ls": "--"}
    ]

    if sbi_samples is not None:
        sbi_gd = MCSamples(
            samples=sbi_samples,
            names=names,
            labels=labels,
            label=r"SBI MAF: $N_{\rm train}=523{,}788$",
            ranges=ranges,
        )
        plot_samples.append(sbi_gd)
        filled.append(True)
        legend_labels.append(
            r"SBI MAF: $N_{\rm train}=523{,}788$"
        )
        contour_colors.append("#C44E52")
        line_args.append(
            {"color": "#C44E52", "lw": 1.3, "ls": "-"}
        )

    for sample in plot_samples:
        sample.updateSettings(
            {
                "smooth_scale_1D": 0.3,
                "smooth_scale_2D": 0.3,
                "fine_bins": 2048,
                "fine_bins_2D": 1024,
            }
        )

    plt.rcParams.update(
        {
            "font.family": "serif",
            "mathtext.fontset": "cm",
            "font.size": 8,
            "axes.labelsize": 8,
            "xtick.labelsize": 7,
            "ytick.labelsize": 7,
            "legend.fontsize": 8,
            "pdf.fonttype": 42,
            "ps.fonttype": 42,
            "savefig.bbox": "tight",
        }
    )

    g = plots.get_subplot_plotter(width_inch=18.0 / 2.54)
    g.settings.axes_fontsize = 7
    g.settings.lab_fontsize = 8
    g.settings.legend_fontsize = 7
    g.settings.alpha_filled_add = 0.28
    g.settings.linewidth = 1.0
    g.settings.num_plot_contours = 2
    g.settings.figure_legend_frame = False
    g.settings.scaling = False

    g.triangle_plot(
        plot_samples,
        params=names,
        filled=filled,
        legend_labels=legend_labels,
        contour_colors=contour_colors,
        line_args=line_args,
        markers=truth,
        marker_args={
            "color": "black",
            "lw": 0.9,
            "ls": ":",
        },
    )

    if sbi_samples is None:
        corner_path = (
            OUTPUT_DIR
            / "fisher_conditional_battaglia12_9param_diagnostic_only.jpg"
        )
        g.fig.suptitle(
            "Diagnostic only: Battaglia12 SBI conditioning "
            "failed joint-support checks",
            color="#8B0000",
            fontsize=9,
            y=0.995,
        )
    else:
        corner_path = (
            OUTPUT_DIR
            / "fisher_conditional_vs_sbi_N523788_"
            "battaglia12_9param.jpg"
        )

    g.fig.savefig(
        corner_path, dpi=300, bbox_inches="tight"
    )
    plt.show()

    print("Status:", comparison_status)
    print("Saved corner:", corner_path)
    print("Saved summary:", summary_csv)
    print("Saved audit:", summary_json_path)
    if sbi_samples is None:
        print(
            "No SBI contour was drawn. Inspect the profile/context "
            "and flow-support diagnostics before generating a "
            "replacement Battaglia12 observation."
        )


## Interpretation

- This is a **conditional-noise** Fisher forecast. It conditions on the fixed HalfDome signal realization and mask, so it does not include halo/sample variance from changing the sky realization.
- The NPE was trained on the consolidated noisy spectra and is evaluated at the exact same independent binned $D_\ell$ vector used by Fisher.
- Fisher is a local linear-Gaussian approximation around Battaglia12. SBI may be non-Gaussian and bounded by the prior.
- A zero or tiny direct-MAF prior acceptance is not a plotting issue. It means the conditioning vector drives the learned flow outside its training/prior support, usually because the observation simulator/preprocessing contract is mismatched or the observation is jointly out of distribution.
- The output summary records the context percentile, OAS shrinkage, Fisher rank, importance ESS, and SBI direct acceptance so the plot remains auditable.

## How to interpret zero acceptance

The notebook distinguishes three possibilities:

1. **Estimator or sampling-API failure:** raw flow acceptance is also poor at saved dataset or training contexts.
2. **Single noise outlier:** controls are healthy, but only the selected Battaglia12 seed fails.
3. **Systematic observation-contract mismatch:** controls are healthy while all 64 Battaglia12 seeds fail and the 40-dimensional context is jointly outside the training distribution.

Only the first case is repaired by changing how the estimator is loaded or sampled. The third requires identifying the simulator difference and generating a matched observation. Disabling hard-prior rejection or switching to MCMC would sample an extrapolative, extremely low-density tail of the NPE and is not a valid repair.